# I modelli di riconoscimento

Il codice del capitolo [«I modelli di riconoscimento»](https://book.paithon.it/main/SpeechRecognition/modelli-asr.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## I modelli di riconoscimento

[Leggi la pagina](https://book.paithon.it/main/SpeechRecognition/modelli-asr.html)


### Dalla rete alla frase: la decodifica


In [ ]:
import itertools

VUOTO = "∅"
# i voti della rete: due frame, due simboli
voti = [{VUOTO: 0.6, "A": 0.4},
        {VUOTO: 0.6, "A": 0.4}]

def collassa(percorso):
    """Prima unisce i simboli uguali consecutivi, poi toglie i vuoti."""
    uniti = [s for i, s in enumerate(percorso)
             if i == 0 or s != percorso[i - 1]]
    return "".join(s for s in uniti if s != VUOTO)

def probabilita(percorso):
    p = 1.0
    for t, s in enumerate(percorso):
        p *= voti[t][s]
    return p

# tutte le trascrizioni, con la somma dei percorsi che le producono
totali = {}
for percorso in itertools.product(VUOTO + "A", repeat=len(voti)):
    testo = collassa(percorso)
    totali[testo] = totali.get(testo, 0.0) + probabilita(percorso)

# il percorso migliore: il simbolo più votato a ogni frame
migliore = tuple(max(v, key=v.get) for v in voti)

print("percorso migliore:", "".join(migliore),
      "->", repr(collassa(migliore)), round(probabilita(migliore), 3))
for testo, p in sorted(totali.items(), key=lambda kv: -kv[1]):
    print(f"  p(y = {testo!r}) = {p:.2f}")

# la promessa del testo, resa eseguibile
assert totali["A"] > totali[collassa(migliore)]

### Misurare gli errori: il Word Error Rate


In [ ]:
import numpy as np

def wer(rif, ip):
    r, h = rif.split(), ip.split()
    # matrice di distanza di edit riempita per programmazione dinamica
    D = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    D[:, 0] = np.arange(len(r) + 1)   # cancellazioni pure
    D[0, :] = np.arange(len(h) + 1)   # inserzioni pure
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            costo = 0 if r[i - 1] == h[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1,          # cancellazione
                          D[i, j - 1] + 1,          # inserzione
                          D[i - 1, j - 1] + costo)  # sostituzione (o parola uguale)
    return D[len(r), len(h)] / len(r)

print(round(wer("il gatto nero salta sul muro",
                "il gatto nemo salta muro"), 3))
# -> 0.333  (1 sostituzione + 1 cancellazione su 6 parole)

## La voce sintetica: dal testo al parlato

[Leggi la pagina](https://book.paithon.it/main/SpeechRecognition/sintesi-vocale.html)


### In pratica: sintetizzare una frase


In [ ]:

import torch
import torchaudio
import soundfile as sf

# bundle preaddestrato: Tacotron 2 (da caratteri) + vocoder WaveRNN,
# voce inglese femminile (dataset LJSpeech)
bundle = torchaudio.pipelines.TACOTRON2_WAVERNN_CHAR_LJSPEECH

processor = bundle.get_text_processor()   # testo -> ID dei caratteri
tacotron2 = bundle.get_tacotron2()        # caratteri -> mel-spettrogramma
vocoder = bundle.get_vocoder()            # mel-spettrogramma -> onda

testo = "The black cat jumps on the wall."

with torch.inference_mode():
    token, lunghezze = processor(testo)
    mel, mel_len, _ = tacotron2.infer(token, lunghezze)
    onda, onda_len = vocoder(mel, mel_len)

# salva il risultato: onda ha forma (batch, campioni), qui se ne prende il primo
sf.write("gatto.wav", onda[0].cpu().numpy(), vocoder.sample_rate)
print(onda.shape, vocoder.sample_rate)  # es. torch.Size([1, ...]) e 22050